# Blinkit On-Demand: Daily Suggestion Generation

Independent of `Blinkit_actions_llm_marketing_Batch_Api.ipynb` and `Blinkit_actions_llm_marketing.ipynb`.
Never reads from or writes to `Blinkit_actions_llm`.

Runs the SAME rule engine + LLM decision logic as the dashboard's manual
"Generate now" button (both call `ondemand_engine.generate_ondemand_suggestions`),
but loops every active campaign for each brand once a day, and adds one thing
manual on-demand runs also get for free: before deciding, it looks back at the
last ACCEPTED decision for that exact keyword and grades whether it actually
worked (MET / MISSED / PAUSED, using real before/after ROAS) - that grade is
folded into this run's prompt so the model can see whether its own past call
was right, not just what it decided.

Writes to `voylla.blinkit_ondemand_actions` only. Run daily at 12:30 pm IST by Windows Task Scheduler through
`jobs/run_job.bat` (see `jobs/README.md`).


In [ ]:
# The engine code comes from this repo's webapp/ folder - the same code the
# dashboard runs and the tests check. The scheduled job pulls the latest
# main before every run, so a merged change is used by the next run.
# (Credentials still live outside git, in Python_Scripts - db.py reads them there.)
import os
import sys

WEBAPP_DIR = os.environ.get("AI_MARKETING_WEBAPP") or os.path.abspath(os.path.join(os.getcwd(), "..", "webapp"))
sys.path.insert(0, WEBAPP_DIR)

import ondemand_engine as oe
from db import get_engine

print(f"Using engine module from: {oe.__file__}")


In [ ]:
# COST CONTROL: scoped down to just the pilot campaign after a Sonnet-model
# full-catalog run cost ~$96 in two days. Only 296464 (Voylla) runs now.
# Add more (campaign_id, brand) pairs here later, deliberately, one at a time.
SCOPE = [("Voylla", ["296464"])]

In [ ]:
from datetime import datetime

run_started = datetime.now()
summary = []

for brand, campaign_ids in SCOPE:
    print(f"\n{'='*60}\nBRAND: {brand} campaigns: {campaign_ids}\n{'='*60}")
    result = oe.generate_daily_suggestions_for_brand(brand, campaign_ids=campaign_ids)
    summary.append(result)
    print(f"  campaigns run: {result['campaigns_run']} | "
          f"suggestions written: {result['total_suggestions']} | "
          f"failures: {len(result['failures'])}")

run_finished = datetime.now()


In [ ]:
# Summary - check this after every scheduled run.
print(f"Daily on-demand generation: {run_started} -> {run_finished}")
for r in summary:
    print(f"  {r['brand']:10s} campaigns={r['campaigns_run']:>4}  "
          f"suggestions={r['total_suggestions']:>4}  failures={len(r['failures'])}")
    for f in r["failures"]:
        print(f"      FAILED campaign {f['campaign_id']}: {f['error']}")
